<center><h1>Numerical linear algebra</h1></center>
<center><h3>The Successive over-relaxation and Conjugate Gradient methods</h3></center>

- - - - - - - - - - - - - - -

In [1]:
import numpy as np

Consider a symmetric, positive-definite, tri-diagonal system $Ax = b$ of size $n = 100$ for $i, j = 1, \dots, n$ where

<center>$A_{ij} = \begin{cases}
    -1, \space \text{if } j = i - 1 \\
    2 + \frac{i}{10}, \space \text{if } j = i \\
    -1, \space \text{if } j = i + 1 \\
    0, \space \text{otherwise}
    \end{cases}$, $\space \space \space b_{i} = 1 + \frac{i}{20}$</center>
<br>
Use the <b>Conjugate Gradient</b> method with the pre-conditioner $C^{-1} = D^{\frac{1}{2}}$ to solve $Ax = b$ to within a tolerance of $||r^{(k)}||_{\infty} < 10^{-5}$, beginning from an initial guess $x^{(0)} = 0$. Report the approximate solution components $x_{1}^{(k)}, \dots, x_{4}^{(k)}$ and number of iterations $k$ required, and the residual norm $||r^{(k)}||_{\infty}$.

In [2]:
def ConjugateGradient(dim, x, tol = 1e-5):
    A = np.zeros((dim, dim))
    b = [0] * (dim)
    C_inv = np.zeros((dim, dim))

    '''Defining the symmetric, positive-definite, tri-diagonal system above.'''
    for i in range(dim):
        for j in range(dim):
            if (j == i - 1) or (j == i + 1):
                A[i][j] = -1
            elif j == i:
                A[i][j] = (2 + (i + 1) / 10)
            else:
                A[i][j] = 0
        b[i] = (1 + (i + 1) / 20)
    
    for i in range(dim):
        C_inv[i][i] = (1 / np.sqrt(A[i][i]))
        # C_inv[i][i] = 1 # if there is no pre-conditioning
    
    '''The solution is then obtained iteratively as:'''
    '''r_0 := b - Ax_0, p_0 := r_0 and k := 0 if r_0 is sufficiently small, then iterate where'''
    '''alpha_k := (r_k^T * r_k)/(p_k^T * Ap_k)'''
    '''x_k+1 := x_k + alpha_k * p_k'''
    '''r_k+1 := r_k - alpha_k * Ap_k until r_k+1 is sufficiently small'''
    '''beta_k := (r_k+1^T * r_k+1)/(r_k^T * r_k)'''
    '''p_k+1 := r_k+1 + beta_k * p_k'''
    '''where Ap_k, alpha_k * p_k, and beta_k * p_k are all calculated through the dot product.'''
    results = []
    iter = 0
    C_invT = np.transpose(C_inv)

    r = b - (A @ x)
    w = C_inv @ r
    v = C_invT @ w
    alpha = np.dot(w, w)

    while (iter < 500):
        u = A @ v
        t = alpha / (np.dot(v, u))
        x = x + (t * v)
        r = r - (t * u)
        w = C_inv @ r
        beta = np.dot(w, w)
        s = beta / alpha
        v_next = (C_invT @ w) + (s * v)
        alpha = beta
        
        error = np.linalg.norm(v - v_next)
        if error < tol:
            results.append(x)
            return results, iter, error
        v = v_next
        iter += 1
        
    return results, iter, error

if __name__ == "__main__":
    dim = 100
    guess = [0] * (dim)
    
    solution = ConjugateGradient(dim, x = guess)

    print(f'Solution converged at |r|_(infinity) = {solution[2]}')
    print(f'Iterations needed: k = {solution[1]}')
    print(f'Approximate solution: x**(k) = {solution[0]}')

Solution converged at |r|_(infinity) = 4.6038339357896e-06
Iterations needed: k = 15
Approximate solution: x**(k) = [array([1.704914  , 2.5303196 , 2.76178968, 2.67179694, 2.45052248,
       2.20450863, 1.98119976, 1.79473059, 1.64404611, 1.52300414,
       1.42496688, 1.34439292, 1.27709081, 1.22000647, 1.17092871,
       1.12824553, 1.09075315, 1.05754146, 1.02790468, 1.00128726,
       0.977245  , 0.95541793, 0.93551094, 0.91727961, 0.90051965,
       0.88505893, 0.87075131, 0.85747189, 0.84511329, 0.83358262,
       0.82279914, 0.81269231, 0.80320024, 0.79426836, 0.78584842,
       0.77789754, 0.77037753, 0.76325424, 0.75649704, 0.75007837,
       0.74397339, 0.73815963, 0.73261671, 0.72732613, 0.72227101,
       0.71743596, 0.71280692, 0.70837097, 0.70411629, 0.70003196,
       0.69610795, 0.69233501, 0.68870456, 0.68520868, 0.68184002,
       0.67859176, 0.67545754, 0.67243147, 0.66950805, 0.66668212,
       0.6639489 , 0.6613039 , 0.6587429 , 0.65626197, 0.65385742,
       0.651

Repeat the above process using the <b>SOR</b> ($\omega = 1.2$) method and briefly compare results. Begin each method from a zero initial guess and use the same tolerance on the residual. Try the <b>SOR</b> method with different $\omega$ values, say $\omega = 1.6$ and $\omega = 2.3$. Does the value of $\omega$ have a significant impact on convergence? Try <b>Conjugate Gradient</b> without pre-conditioning ($C^{-1} = I$). Does pre-conditioning make a noticeable difference?

#### Successive over-relaxation method
($\omega = 1.2$)

In [3]:
def SOR(dim, x, w, tol = 1e-5):
    A = np.zeros((dim, dim))
    b = [0] * (dim)
    C_inv = np.zeros((dim, dim))

    '''Defining the symmetric, positive-definite, tri-diagonal system above.'''
    for i in range(dim):
        for j in range(dim):
            if (j == i - 1) or (j == i + 1):
                A[i][j] = -1
            elif j == i:
                A[i][j] = (2 + (i + 1) / 10)
            else:
                A[i][j] = 0
        b[i] = (1 + (i + 1) / 20)

    '''This will let us isolate the diagonal of the matrix and turn it into its own matrix.'''
    C = np.diag(A)
    D = np.diagflat(C)

    '''This will construct the pre-conditioned matrix.'''
    for i in range(dim):
        C_inv[i][i] = (1 / np.sqrt(D[i][i]))
        # C_inv[i][i] = 1 # if there is no pre-conditioning
    C = C_inv
    
    '''This gives us the strictly upper and lower matrix components of A.'''
    lower = D - np.tril(A)
    upper = D - np.triu(A)
    
    '''With one of the main components being (D - wL)**-1, this will get the inverse of the matrix.'''
    DwL_inv = np.linalg.inv(D - w * lower)

    '''The solution is then obtained iteratively as:'''
    '''x**(k+1) = (D - wL)**-1 * [wU + (1 - w)D]x + w(D - wL)**-1(b)'''
    '''where [wU + (1 - w)D]x and w(D - wL)**-1(b) are all calculated through the dot product.'''
    results = []
    iter = 0
    
    while (iter < 500):
        x_next = DwL_inv @ ((w * upper + (1 - w) * D) @ x) + (w * DwL_inv @ b)
        error = np.linalg.norm(x - x_next)
        if error < tol:
            results.append(x_next)
            return results, iter, error
        x = x_next
        iter += 1
        
    return results, iter, error

if __name__ == "__main__":
    dim = 100
    guess = [0] * (dim)
    
    solution = SOR(dim, w = 1.2, x = guess)

    print(f'Solution converged at |r|_(infinity) = {solution[2]}')
    print(f'Iterations needed: k = {solution[1]}')
    print(f'Approximate solution: x**(k) = {solution[0]}')

Solution converged at |r|_(infinity) = 3.901534967913226e-06
Iterations needed: k = 17
Approximate solution: x**(k) = [array([1.70491285, 2.53031856, 2.76178879, 2.67179598, 2.45052167,
       2.2045082 , 1.98119964, 1.79473083, 1.64404668, 1.52300454,
       1.42496693, 1.34439294, 1.27709048, 1.22000566, 1.17092875,
       1.12824496, 1.09075312, 1.05754159, 1.02790491, 1.00128755,
       0.97724531, 0.9554182 , 0.93551115, 0.91727974, 0.90051969,
       0.88505889, 0.87075119, 0.85747173, 0.84511309, 0.8335824 ,
       0.82279892, 0.81269211, 0.80320006, 0.79426822, 0.78584831,
       0.77789746, 0.77037749, 0.76325423, 0.75649706, 0.75007842,
       0.74397347, 0.73815972, 0.73261682, 0.72732624, 0.72227113,
       0.71743609, 0.71280704, 0.70837109, 0.7041164 , 0.70003206,
       0.69610805, 0.6923351 , 0.68870464, 0.68520874, 0.68184007,
       0.67859179, 0.67545757, 0.67243149, 0.66950805, 0.66668212,
       0.66394889, 0.66130387, 0.65874287, 0.65626193, 0.65385737,
       0.6

#### Successive over-relaxation method
($\omega = 1.6$)

In [4]:
def SOR(dim, x, w, tol = 1e-5):
    A = np.zeros((dim, dim))
    b = [0] * (dim)
    C_inv = np.zeros((dim, dim))

    '''Defining the symmetric, positive-definite, tri-diagonal system above.'''
    for i in range(dim):
        for j in range(dim):
            if (j == i - 1) or (j == i + 1):
                A[i][j] = -1
            elif j == i:
                A[i][j] = (2 + (i + 1) / 10)
            else:
                A[i][j] = 0
        b[i] = (1 + (i + 1) / 20)

    '''This will let us isolate the diagonal of the matrix and turn it into its own matrix.'''
    C = np.diag(A)
    D = np.diagflat(C)

    '''This will construct the pre-conditioned matrix.'''
    for i in range(dim):
        C_inv[i][i] = (1 / np.sqrt(D[i][i]))
        # C_inv[i][i] = 1 # if there is no pre-conditioning
    C = C_inv
    
    '''This gives us the strictly upper and lower matrix components of A.'''
    lower = D - np.tril(A)
    upper = D - np.triu(A)
    
    '''With one of the main components being (D - wL)**-1, this will get the inverse of the matrix.'''
    DwL_inv = np.linalg.inv(D - w * lower)

    '''The solution is then obtained iteratively as:'''
    '''x**(k+1) = (D - wL)**-1 * [wU + (1 - w)D]x + w(D - wL)**-1(b)'''
    '''where [wU + (1 - w)D]x and w(D - wL)**-1(b) are all calculated through the dot product.'''
    results = []
    iter = 0
    
    while (iter < 500):
        x_next = DwL_inv @ ((w * upper + (1 - w) * D) @ x) + (w * DwL_inv @ b)
        error = np.linalg.norm(x - x_next)
        if error < tol:
            results.append(x_next)
            return results, iter, error
        x = x_next
        iter += 1
        
    return results, iter, error

if __name__ == "__main__":
    dim = 100
    guess = [0] * (dim)
    
    solution = SOR(dim, w = 1.6, x = guess)

    print(f'Solution converged at |r|_(infinity) = {solution[2]}')
    print(f'Iterations needed: k = {solution[1]}')
    print(f'Approximate solution: x**(k) = {solution[0]}')

Solution converged at |r|_(infinity) = 8.571881101366606e-06
Iterations needed: k = 26
Approximate solution: x**(k) = [array([1.70491424, 2.5303193 , 2.76178865, 2.6717961 , 2.45052242,
       2.20450891, 1.98119984, 1.79473074, 1.64404663, 1.52300454,
       1.4249669 , 1.3443929 , 1.27709048, 1.22000568, 1.17092876,
       1.12824496, 1.09075311, 1.05754159, 1.02790491, 1.00128756,
       0.9772453 , 0.9554182 , 0.93551115, 0.91727974, 0.9005197 ,
       0.88505889, 0.87075119, 0.85747172, 0.84511309, 0.8335824 ,
       0.82279892, 0.81269211, 0.80320006, 0.79426822, 0.78584831,
       0.77789746, 0.77037749, 0.76325423, 0.75649706, 0.75007842,
       0.74397347, 0.73815972, 0.73261682, 0.72732624, 0.72227113,
       0.71743609, 0.71280704, 0.70837109, 0.7041164 , 0.70003206,
       0.69610805, 0.6923351 , 0.68870464, 0.68520874, 0.68184007,
       0.6785918 , 0.67545757, 0.67243149, 0.66950805, 0.66668212,
       0.66394889, 0.66130387, 0.65874287, 0.65626194, 0.65385737,
       0.6

#### Successive over-relaxation method
($\omega = 2.3$)

In [5]:
def SOR(dim, x, w, tol = 1e-5):
    A = np.zeros((dim, dim))
    b = [0] * (dim)
    C_inv = np.zeros((dim, dim))

    '''Defining the symmetric, positive-definite, tri-diagonal system above.'''
    for i in range(dim):
        for j in range(dim):
            if (j == i - 1) or (j == i + 1):
                A[i][j] = -1
            elif j == i:
                A[i][j] = (2 + (i + 1) / 10)
            else:
                A[i][j] = 0
        b[i] = (1 + (i + 1) / 20)

    '''This will let us isolate the diagonal of the matrix and turn it into its own matrix.'''
    C = np.diag(A)
    D = np.diagflat(C)

    '''This will construct the pre-conditioned matrix.'''
    for i in range(dim):
        C_inv[i][i] = (1 / np.sqrt(D[i][i]))
        # C_inv[i][i] = 1 # if there is no pre-conditioning
    C = C_inv
    
    '''This gives us the strictly upper and lower matrix components of A.'''
    lower = D - np.tril(A)
    upper = D - np.triu(A)
    
    '''With one of the main components being (D - wL)**-1, this will get the inverse of the matrix.'''
    DwL_inv = np.linalg.inv(D - w * lower)

    '''The solution is then obtained iteratively as:'''
    '''x**(k+1) = (D - wL)**-1 * [wU + (1 - w)D]x + w(D - wL)**-1(b)'''
    '''where [wU + (1 - w)D]x and w(D - wL)**-1(b) are all calculated through the dot product.'''
    results = []
    iter = 0
    
    while (iter < 500):
        x_next = DwL_inv @ ((w * upper + (1 - w) * D) @ x) + (w * DwL_inv @ b)
        error = np.linalg.norm(x - x_next)
        if error < tol:
            results.append(x_next)
            return results, iter, error
        x = x_next
        iter += 1
        
    return results, iter, error

if __name__ == "__main__":
    dim = 100
    guess = [0] * (dim)
    
    solution = SOR(dim, w = 2.3, x = guess)

    print(f'Solution converged at |r|_(infinity) = {solution[2]}')
    print(f'Iterations needed: k = {solution[1]}')
    print(f'Approximate solution: x**(k) = {solution[0]}')

Solution converged at |r|_(infinity) = 1.929856531157443e+62
Iterations needed: k = 500
Approximate solution: x**(k) = []


In the <b>Conjugate Gradient</b> method, pre-conditioning changed the amount of iterations it needed to converge. This was not exactly true for the <b>Successive over-relaxation</b> method. The biggest difference maker was changing the weights. Changing the weights really impacted its convergence, and when $\omega = 1.2$ jumped to $\omega = 1.6$, that added almost another 10 iterations to converging. When that weight was changed to $\omega = 2.3$, it would not even converge after 500 iterations.

## Further readings:
Reading material:<br>
https://www.amazon.com/Numerical-Analysis-Richard-L-Burden/dp/1305253663/

The @ symbol: <br>
https://stackoverflow.com/questions/6392739/what-does-the-at-symbol-do-in-python

## Useful links:
Quick definitions:<br>
https://mathworld.wolfram.com/SuccessiveOverrelaxationMethod.html <br>
https://mathworld.wolfram.com/ConjugateGradientMethod.html